# Imports

In [ ]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, row_number, concat_ws, length

# Connect to Spark

In [ ]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use.

In [ ]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=400)

# Define global variables and functions

In [ ]:
airports = [
    "KADW", "KATL", "KBOS", "KBWI", "KCLT", "KDCA", "KDEN", "KDFW", "KDTW",
    "KEWR", "KFLL", "KIAD", "KIAH", "KJFK", "KLAS", "KLAX", "KLGA", "KMCO",
    "KMDW", "KMEM", "KMIA", "KMSP", "KORD", "KPHL", "KPHX", "KSAN", "KSDF",
    "KSEA", "KSFO", "KSLC", "KTPA", "PANC", "PHNL",
]

In [ ]:
%run ./shared_variables.ipynb
%run ./S3_csv_writers.ipynb

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}
print(dates)

In [ ]:
print("retrieving procedures: ", datetime.now())

# Retrieve SIDs and STARs for the airports of interest

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [ ]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .withColumn("sorted_path_terminators", F.concat_ws(":", F.array_sort(F.split(F.col("path_terminators"), ":"))))
    .withColumn("end_date", F.col("metadata.effective_end_date"))
    .withColumn("transition_name", F.coalesce(F.col("transition_name"), F.lit("NONE")))
    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), "path_terminators"))

#    prefix the approach procedure_name with the airport since the procedure_name isn't unique across all airports; e.g. many airports have "ILS OR LOC RWY 10"
    .withColumn("procedure_name", when((F.col("procedure_type") == "APPROACH"), concat_ws("_", F.col("airport.name"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))

    .filter(F.col("procedure_type").isin("SID", "STAR", "APPROACH"))
    .filter(F.col("dtpp_procedure_name").isNotNull())

#    .filter(F.col("airport.icao_region").startswith("K") | F.col("airport.icao_region").startswith("PA") | F.col("airport.icao_region").startswith("PH"))
#    .filter(F.col("procedure_name") == "CAVLR6")
#    .filter(F.col("procedure_name") == "DOCCS3")
#    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("transition_name") == "ALL")
#    .filter(F.col("metadata.effective_end_date") == 1769040000000)
#    .filter(F.col("partitions").isNotNull())

#    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))

#    .filter(F.col("procedure_name") == "APRES2")
#    .filter(F.col("airport").isin(airports))

#    .filter(F.col("airport.name") == "KBWI")
#    .filter(F.col("procedure_type") == "STAR")
#    .filter(F.col("procedure_name").startswith("NUBER6"))
#    .filter(F.col("procedure_name") == "HUDZY5")
#    .filter(F.col("dtpp_procedure_name") == "ILS OR LOC RWY 08L")

#    .filter(F.col("dtpp_procedure_name") == "ILS OR LOC RWY 08R")
#    .filter(F.col("dtpp_procedure_name") == "RNAV (RNP) Y RWY 26L")
#    .filter(F.col("dtpp_procedure_name") == "GLS RWY 26L")
        
    .drop("arinc_record_info", "arinc_route_type_qualifier_1", "arinc_route_type_qualifier_2", "arinc_route_type", "recommended_navaids", "alternative_dtpp_keys", "alternative_dtpp_procedure_names")
    .persist()
)

In [ ]:
# df_procedures.show()

In [ ]:
# get the most recent rows
star_max_end_date = df_procedures.filter(F.col("procedure_type") == "STAR").select(F.max("end_date")).first()[0]
sid_max_end_date = df_procedures.filter(F.col("procedure_type") == "SID").select(F.max("end_date")).first()[0]
iap_max_end_date = df_procedures.filter(F.col("procedure_type") == "APPROACH").select(F.max("end_date")).first()[0]

In [ ]:
print("star_max_end_date: " + datetime.fromtimestamp(star_max_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))
print("sid_max_end_date: " + datetime.fromtimestamp(sid_max_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))
print("iap_max_end_date: " + datetime.fromtimestamp(iap_max_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# # get the newest entry based on the end_date

# window = Window.partitionBy("procedure_name").orderBy(col("end_date").desc())

# df_newest_iaps = (df_procedures
#     .filter(F.col("procedure_type") == "APPROACH")
#     .withColumn("row", row_number().over(window))
#     .filter(col("row") == 1)
#     .drop("row")
# #    .orderBy("procedure_name")
#     .persist()
# )

In [ ]:
# df_newest_iaps.filter(F.col("procedure_name").startswith("2V5_")).show()

In [ ]:
# get the latest STARs
df_newest_stars = (df_procedures
    .filter(F.col("procedure_type") == "STAR")
    .filter(F.col("end_date") == star_max_end_date)
#    .orderBy("grouping")
    .persist()
)

# get the latest SIDs
df_newest_sids = (df_procedures
    .filter(F.col("procedure_type") == "SID")
    .filter(F.col("end_date") == sid_max_end_date)
#    .orderBy("grouping")
    .persist()
)

df_newest_iaps = (df_procedures
    .filter(F.col("procedure_type") == "APPROACH")
    .filter(F.col("end_date") == iap_max_end_date)
    .orderBy("grouping")
    .persist()
)

df_procedures_newest = df_newest_stars.unionByName(df_newest_sids).unionByName(df_newest_iaps).drop("grouping", "metadata")

In [ ]:
# df_procedures_newest.show()

In [ ]:
# df_newest_iaps.filter(F.col("procedure_name") == "00R_R30").show()
# df_procedures_newest.filter(F.col("procedure_name").startswith("2V5_")).show()
# df_procedures_newest.filter(F.col("procedure_type") == "APPROACH").show()

In [ ]:
# # get the most recent rows
# star_end_date = df_procedures.filter(F.col("procedure_type") == "STAR").select(F.max("end_date")).first()[0]
# print("star_end_date: " + datetime.fromtimestamp(star_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))

# sid_end_date = df_procedures.filter(F.col("procedure_type") == "SID").select(F.max("end_date")).first()[0]
# print("sid_end_date: " + datetime.fromtimestamp(sid_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))

# iap_end_date = df_procedures.filter(F.col("procedure_type") == "APPROACH").select(F.max("end_date")).first()[0]
# print("iap_end_date: " + datetime.fromtimestamp(iap_end_date / 1000).strftime("%Y-%m-%d %H:%M:%S"))

# df_procedures_newest = (df_procedures
#     .filter(F.col("end_date") == max_value)
#     .orderBy("grouping")
#     .persist()
# )

In [ ]:
# df_procedures_newest.filter(F.col("procedure_type") == "STAR").show()

In [ ]:
# df_procedures_newest.count()

In [ ]:
# df_procedures_newest.show()

In [ ]:
## get the newest entry based on the end_date
#
# window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#
# df_procedures_newest = (df_procedures
#     .withColumn("row", row_number().over(window))
#     .filter(col("row") == 1)
#     .drop("row")
#     .orderBy("grouping")
#     .persist()
# )

In [ ]:
# df_procedures_newest.count()

In [ ]:
# df_procedures_newest.show()

In [ ]:
# df_procedures_newest.orderBy(F.col("end_date").desc()).show()

In [ ]:
## date of the latest data
#epoch_ms = df_procedures_newest.first()["end_date"]
#print(datetime.fromtimestamp(epoch_ms / 1000).strftime("%Y-%m-%d %H:%M:%S"))
#df_procedures_newest = df_procedures_newest.drop("end_date")

In [ ]:
#df_procedures_newest.select("path_terminators").show(df_procedures.count(), truncate=False)
#df_procedures_newest.filter(F.col("primary_key") == '4S5N2h7TvDA-0_L').select("legs").show(truncate=False)
#df_procedures_newest.show()

In [ ]:
# df_procedures_newest.orderBy(["navigation_source", "procedure_name", "transition_name", "sorted_path_terminators"]).show()

In [ ]:
# duplicates = (
#     df_procedures_newest.withColumn("_duplicate_count", F.count("*").over(Window.partitionBy("navigation_source", "procedure_name", "transition_name", "sorted_path_terminators")))
#       .filter(F.col("_duplicate_count") > 1)
#       .drop("_duplicate_count")
# )

In [ ]:
# duplicates.count()

In [ ]:
# duplicates.show()

In [ ]:
# remove duplicates that are present because the same procedure is used for different airports
df_procedures_nodups = df_procedures_newest.dropDuplicates(["navigation_source", "procedure_name", "transition_name", "sorted_path_terminators"]).orderBy(["navigation_source", "procedure_name", "transition_name", "sorted_path_terminators"])
df_procedures_nodups.count()

In [ ]:
# hack to just use one navigation_source and to prefer CIPF over LIDO if there are two.

df_available_nav_sources = (
    df_procedures_nodups
    .select(
        "procedure_name",
        "navigation_source",
    )
    .distinct()
    .orderBy("procedure_name")
)

window = Window.partitionBy("procedure_name").orderBy(col("navigation_source"))
df_available_nav_sources = (df_available_nav_sources
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)

df_procedures_nodups = df_procedures_nodups.join(df_available_nav_sources, on=["procedure_name", "navigation_source"], how="left_semi")

In [ ]:
df_procedures_nodups.count()

In [ ]:
# df_procedures_nodups.show()

In [ ]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures_exploded = (
    df_procedures_nodups
    .select(
        F.col("primary_key").alias("segment_id"),
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "transition_name",
        "transition_type",
        "navigation_source",
#        "route_type",
        F.col("airport.icao_region").alias("airport_icao_region"),
        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
#        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("fix_name", F.col("leg.path_terminator.identification.name"))
    .withColumn("fix_icao_region", F.col("leg.path_terminator.identification.icao_region"))
    .withColumn("seq", F.col("leg.sequence_number").alias("seq"))
    .withColumn("latitude", F.col("leg.path_terminator.latitude"))
    .withColumn("longitude", F.col("leg.path_terminator.longitude"))
    .withColumn("magnetic_variation", F.col("leg.path_terminator.magnetic_variation.published"))
    .withColumn("speed_description", F.col("leg.speed_limit.descriptor"))
    .withColumn("speed_limit", F.col("leg.speed_limit.limit"))
    .withColumn("speed_altitude", F.col("leg.speed_limit.altitude"))
    .withColumn("altitude_description", F.col("leg.altitude_limits.description"))
    .withColumn("altitude_value1", F.col("leg.altitude_limits.value1"))
    .withColumn("altitude_value2", F.col("leg.altitude_limits.value2"))

#    .withColumn("leg_type", F.col("leg.leg_type"))
#    .withColumn("turn_direction", F.col("leg.turn_direction"))
#    .withColumn("arc_radius", F.col("leg.arc_radius"))
#    .withColumn("rnp", F.col("leg.fix_guidance.rnp"))
#    .withColumn("inbound_magnetic_course", F.col("leg.fix_guidance.inbound_magnetic_course"))
#    .withColumn("outbound_magnetic_course", F.col("leg.fix_guidance.outbound_magnetic_course"))
        
    # prefix the fix_name with the airport when it is the runway fix on an approach since the bare fix_name isn't unique across all airports; e.g. many airports have "RW08"
    # 20260922: previously i was trying to be surgical about which fixnames to tweak, but i think it's better to just prefix all the approach fixes with the airport
    # 20260924: i'm going to try a more gentle touch and only prefix fixes that contain numbers
#    .withColumn("fix_name", when(((F.col("procedure_type") == "APPROACH") & F.col("fix_name").startswith("RW")), concat_ws("_", F.col("airport"), F.col("fix_name"))).otherwise(F.col("fix_name")))
#    .withColumn("fix_name", when(((F.col("procedure_type") == "APPROACH") & (F.col("fix_name").rlike(r"^[A-Za-z]{1,2}[0-9]{2}[LCR]?[A-Za-z]{0,2}$") | (F.length(F.col("fix_name")) == 2) | (F.length(F.col("fix_name")) == 4))), concat_ws("_", F.col("airport"), F.col("fix_name"))).otherwise(F.col("fix_name")))
#    .withColumn("fix_name", when((F.col("procedure_type") == "APPROACH"), concat_ws("_", F.col("airport"), F.col("fix_name"))).otherwise(F.col("fix_name")))
    .withColumn("fix_name", when(((F.col("procedure_type") == "APPROACH") & F.col("fix_name").rlike(r"[0-9]+")), concat_ws("_", F.col("airport"), F.col("fix_name"))).otherwise(F.col("fix_name")))
        
    .filter(F.col("procedure_type").isin("SID", "STAR", "APPROACH"))
#    .filter(F.col("transition_type") != "MISSED")

#    prefix the approach procedure_name with the airport since the procedure_name isn't unique across all airports; e.g. many airports have "ILS OR LOC RWY 10"
#    .withColumn("procedure_name", when(F.col("procedure_type") == "APPROACH", concat_ws("_", F.col("airport"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))

#    .withColumn("transition_name", F.coalesce(F.col("transition_name"), F.lit("n/a")))
#    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
#    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
        
    .drop("leg")
        
#    .dropDuplicates(["path_terminators"])
#    .orderBy("grouping", col("end_date").desc())
    .persist()
)

df_procedures_exploded = (
    df_procedures_exploded
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("latitude").isNotNull())
    .filter(F.col("longitude").isNotNull())
    .filter(F.col("magnetic_variation").isNotNull())
 
#    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))

#    .filter(F.col("navigation_source").isin(["CIFP", "LIDO"]))
#    .filter(F.col("end_date") == 1769040000000)
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("navigation_source") == "LIDO")

#    .filter(F.col("procedure_name") == "LGTNG5")
#    .filter(F.col("procedure_name") == "BNFSH2")
#    .filter(F.col("procedure_name") == "ATL2")
#    .filter(F.col("procedure_name") == "SCOT7")
#    .filter(F.col("procedure_name") == "CAVLR6")
#    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("procedure_name") == "SUNSS8")
#    .filter(F.col("procedure_name").isin(["GIBBZ5", "CAVLR6"]))

#    .filter(F.col("fix_name") == "MEEGO")

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports))

#    .filter(F.col("grouping").isNotNull())
#    .filter(F.col("transition_name").isNotNull())
#    .filter(F.col("fix_name").isNotNull())
    .persist()
)

In [ ]:
df_procedures_exploded.count()

In [ ]:
# df_procedures_exploded.show(df_procedures_exploded.count())
# df_procedures_exploded.filter(F.col("procedure_name") == "KADW_I01L").show()
# df_procedures_exploded.filter(F.col("fix_name") == "LARII").show()
# df_procedures_exploded.show()

In [ ]:
# df_procedures_exploded_output = (
#     df_procedures_exploded
#         .select(
#             "dtpp_procedure_name",
#             "navigation_source",
#             "procedure_name",
#             "procedure_type",
#             "transition_name",
#             "segment_id",
#             "transition_type",
#             "fix_name",
#             "fix_icao_region",
#             "seq",
#             "latitude",
#             "longitude",
#             "magnetic_variation",
# #            "airport",
#             "speed_description",
#             "speed_limit",
#             "speed_altitude",
#             "altitude_description",
#             "altitude_value1",
#             "altitude_value2",
#         )
# #        .coalesce(1)
#         .withColumn("proc_type", F.col("procedure_type"))
#         .withColumn("proc_name", F.col("procedure_name"))
#         .repartition("proc_type", "proc_name")
# )

In [ ]:
# (
#     df_procedures_exploded_output
#         .repartition(1)
#         .write.option("header", True)
#         .partitionBy(["proc_type", "proc_name"])
#         .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
# )

In [ ]:
# spark_write_cre(df_procedures_exploded_output, path="CRAFT/" + year0 + "/procedures", writer=write_csv_with_partitions, partitionBy=["proc_type", "proc_name"], compression=None, mode="overwrite")

In [ ]:
# (
#     df_procedures_exploded
#         .select(
#             "dtpp_procedure_name",
#             "navigation_source",
#             "procedure_name",
#             "procedure_type",
#             "transition_name",
#             "segment_id",
#             "transition_type",
#             "fix_name",
#             "fix_icao_region",
#             "seq",
#             "latitude",
#             "longitude",
#             "magnetic_variation",
# #            "airport",
#             "speed_description",
#             "speed_limit",
#             "speed_altitude",
#             "altitude_description",
#             "altitude_value1",
#             "altitude_value2",
#         )
#         .coalesce(1)
#         .write.option("header", True)
#         .csv("CRAFT/" + year0 + "/procedures/all_sidstar", compression="None", mode="overwrite")
# )

In [ ]:
df_procedures_output = (
    df_procedures_exploded
        .select(
#            "airport",
            "dtpp_procedure_name",
            "navigation_source",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "segment_id",
            "transition_type",
            "fix_name",
            "fix_icao_region",
            "seq",
            "latitude",
            "longitude",
            "magnetic_variation",
            "speed_description",
            "speed_limit",
            "speed_altitude",
            "altitude_description",
            "altitude_value1",
            "altitude_value2",
        )
)

In [ ]:
# df_procedures_output.filter(F.col("procedure_type") == "APPROACH").show()

In [ ]:
(
    df_procedures_output
        .filter(F.col("procedure_type") == "APPROACH")
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_iap", compression="None", mode="overwrite")
)

(
    df_procedures_output
        .filter(F.col("procedure_type") == "STAR")
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_star", compression="None", mode="overwrite")
)

(
    df_procedures_output
        .filter(F.col("procedure_type") == "SID")
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_sid", compression="None", mode="overwrite")
)

In [ ]:
# spark_write_cre(df_procedures_sidstar_output, path="CRAFT/" + year0 + "/procedures/all_sidstar", writer=write_csv, compression=None, mode="overwrite")

## output the procedures (SIDs and STARs) for each airport

In [ ]:
# # v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# # https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

# df_procedures_by_airports = (
#     api.dataframe("ArincTransition", **dates, metadata=True)
#     .select(
#         "dtpp_procedure_name",
#         "procedure_name",
#         "procedure_type",
#         "navigation_source",
#         F.col("airport.icao_region").alias("airport_icao_region"),
#         F.col("airport.name").alias("airport"),
#         F.col("metadata.effective_end_date").alias("end_date")
#     )
#     .withColumn("grouping", F.concat("airport", F.lit("_"), "procedure_name"))
        
#     # prefix the approach procedure_name with the airport since the procedure_name isn't unique across all airports; e.g. many airports have "ILS OR LOC RWY 10"
#     # .withColumn("procedure_name", when(((F.col("procedure_type") == "APPROACH") | (F.col("procedure_type") == "MISSED")) , concat_ws("_", F.col("airport.name"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))
#     .withColumn("procedure_name", when((F.col("procedure_type") == "APPROACH"), concat_ws("_", F.col("airport"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))
        
# #    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))
#     .filter(F.col("procedure_type").isin("SID", "STAR", "APPROACH"))

# #    comment the following out so we get procedures for all US airports
# #    .filter(F.col("airport").isin(airports)) 
# #    .filter(F.col("airport") == "KIAD")
# #    .filter(F.col("airport") == "ENVA")
#     .filter(F.col("dtpp_procedure_name").isNotNull())

#     .orderBy("grouping")
#     .persist()
# )

In [ ]:
df_procedures_newest.filter(F.col("procedure_name").startswith("2V5_")).show()

In [ ]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures_by_airports_newest = (
    df_procedures_newest
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "navigation_source",
        F.col("airport.icao_region").alias("airport_icao_region"),
        F.col("airport.name").alias("airport"),
#        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("grouping", F.concat("airport", F.lit("_"), "procedure_name"))
        
    # prefix the approach procedure_name with the airport since the procedure_name isn't unique across all airports; e.g. many airports have "ILS OR LOC RWY 10"
    # .withColumn("procedure_name", when(((F.col("procedure_type") == "APPROACH") | (F.col("procedure_type") == "MISSED")) , concat_ws("_", F.col("airport.name"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))
#    .withColumn("procedure_name", when((F.col("procedure_type") == "APPROACH"), concat_ws("_", F.col("airport"), F.col("procedure_name"))).otherwise(F.col("procedure_name")))
        
#    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))
    .filter(F.col("procedure_type").isin("SID", "STAR", "APPROACH"))

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports)) 
#    .filter(F.col("airport") == "KIAD")
#    .filter(F.col("airport") == "ENVA")
    .filter(F.col("dtpp_procedure_name").isNotNull())

    .orderBy("grouping")
    .persist()
)

In [ ]:
# df_procedures_by_airports.filter(F.col("procedure_name") == "00R_R30").show()
# df_procedures_by_airports.filter(F.col("procedure_type") == "STAR").show()
# df_procedures_newest.filter(F.col("procedure_name").startswith("2V5_")).show()
df_procedures_by_airports_newest.filter(F.col("procedure_name").startswith("2V5_")).show()

In [ ]:
# # get the most recent rows
# max_value = df_procedures.select(F.max("end_date")).first()[0]
# print(datetime.fromtimestamp(max_value / 1000).strftime("%Y-%m-%d %H:%M:%S"))

# df_procedures_by_airports_newest = (df_procedures_by_airports
#     .filter(F.col("end_date") == max_value)
#      .drop("row", "end_date")
#      .orderBy("procedure_type", "dtpp_procedure_name", "grouping")
#     .persist()
# )

In [ ]:
# window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#
# df_procedures_by_airports_newest = (df_procedures_by_airports
#     .withColumn("row", row_number().over(window))
#     .filter(col("row") == 1)
#     .drop("row", "end_date")
#     .orderBy("procedure_type", "dtpp_procedure_name", "grouping")
# )

In [ ]:
# df_procedures_by_airports_newest.count()

In [ ]:
#df_procedures_by_airports_newest.show(df_procedures_by_airports_newest.count())
#df_procedures_by_airports_newest.show()

In [ ]:
# (
#     df_procedures_by_airports_newest
#         .select(
#             "airport",
#             "procedure_type",
#             "procedure_name",
#         )
#         .orderBy("airport", "procedure_type", "procedure_name")
#         .write.option("header", True)
#         .csv("CRAFT/" + year0 + "/procedures/airports", compression="None", mode="overwrite")
# )

In [ ]:
df_procedures_by_airports_newest_output = (
    df_procedures_by_airports_newest
        .select(
            "airport",
            "airport_icao_region",
            "procedure_type",
            "procedure_name",
        )
        .dropDuplicates()
        .orderBy("airport", "procedure_type", "procedure_name")
)

In [ ]:
df_procedures_by_airports_newest_output.filter(F.col("procedure_name").startswith("2V5_")).show()

In [ ]:
(
    df_procedures_by_airports_newest_output
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/airports", compression="None", mode="overwrite")
)

In [ ]:
# spark_write_cre(df_procedures_by_airports_newest_output, path="CRAFT/" + year0 + "/procedures/airports", writer=write_csv, compression=None, mode="overwrite")

## output all procedure fixes
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [ ]:
#(
#    df_procedures_exploded.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation")
#        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
#        .orderBy("fix_name")
#        .write.option("header", True)
#        .csv("CRAFT/" + year0 + "/procedures/all_fixes", compression="None", mode="overwrite")
#)

In [ ]:
df_procedures_allfixes_output = (
#    df_procedures_exploded.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation", "navigation_source")
    df_procedures_exploded.groupBy("fix_name", "fix_icao_region")
        .agg(
            F.mean("latitude").alias("latitude"),
            F.mean("longitude").alias("longitude"),
            F.mean("magnetic_variation").alias("magnetic_variation"),
            F.first("navigation_source").alias("navigation_source"),
            F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"),
            F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
        .orderBy("fix_name")
#        .dropDuplicates(["fix_name"])
)

In [ ]:
# df_procedures_exploded.filter(F.col("fix_name") == "LARII").show()

In [ ]:
# df_procedures_allfixes_output.filter(F.col("fix_name") == "KGAO_LEV").show()
# df_procedures_allfixes_output.filter(F.col("fix_name") == "LARII").show()

In [ ]:
(
    df_procedures_allfixes_output
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_fixes", compression="None", mode="overwrite")
)

In [ ]:
# spark_write_cre(df_procedures_allfixes_output, path="CRAFT/" + year0 + "/procedures/all_fixes", writer=write_csv, compression=None, mode="overwrite")